# Lesson 13.2: A shared task board needs a precise claim rule

Two workers can see the same waiting task. Only one should gain ownership. Follow the database condition that checks whether the task is still free at the moment the claim is recorded. Then distinguish a worker's report of completion from acceptance of the work by a receiving team.

[Companion lesson](../modules/13-agent-frameworks/02-code-workshop.md) · [Original-source walkthrough](../modules/13-agent-frameworks/05-source-walkthrough.md)

Run one cell at a time from top to bottom. The programme uses the fictional records shown in the code. No API key or model download is needed.

## Predict the result

The first two lines show the two claims. The third shows the final task status and owner. Predict which worker can report the result.

In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute("""CREATE TABLE tasks (
    id INTEGER PRIMARY KEY,
    status TEXT NOT NULL,
    owner TEXT,
    result TEXT
)""")
connection.execute("INSERT INTO tasks (id, status) VALUES (?, ?)", (17, "pending"))
connection.commit()

def claim(connection, task_id, worker):
    with connection:
        changed = connection.execute(
            "UPDATE tasks SET status = ?, owner = ? WHERE id = ? AND status = ?",
            ("in_progress", worker, task_id, "pending")
        ).rowcount
    return changed == 1

def report(connection, task_id, worker, result):
    with connection:
        changed = connection.execute(
            "UPDATE tasks SET status = ?, result = ? "
            "WHERE id = ? AND owner = ? AND status = ?",
            ("reported_done", result, task_id, worker, "in_progress")
        ).rowcount
    if changed != 1:
        raise ValueError("Task is not held by this worker in the expected state")

first_claim = claim(connection, 17, "worker-a")
second_claim = claim(connection, 17, "worker-b")
report(connection, 17, "worker-a", "Draft prepared for review")
final_task = connection.execute("SELECT status, owner FROM tasks WHERE id = 17").fetchone()
print(first_claim, second_claim, final_task, sep="\n")
connection.close()

## Inspect the result

`print` displays values below the cell. When present, `assert` checks a specific expectation and raises an error if it fails.

In [ ]:
assert first_claim is True
assert second_claim is False
assert final_task == ("reported_done", "worker-a")

## Try a change

Before running again, predict the owner if worker-b claims first. Explain what should happen if worker-a then tries to report.

Make a prediction before opening the worked answer.

In [ ]:
# Try the change here, or make a temporary edit in the first code cell.

<details>
<summary>Worked answer</summary>

Worker-b would hold the task. Worker-a's report should be refused unless the caller passes the expected ownership check. The final status records a worker report, not business acceptance.

```python
with sqlite3.connect(":memory:") as board:
    board.execute("CREATE TABLE tasks (id, status, owner, result)")
    board.execute("INSERT INTO tasks (id, status) VALUES (17, 'pending')")
    assert claim(board, 17, "worker-b")
    assert not claim(board, 17, "worker-a")
    try:
        report(board, 17, "worker-a", "Draft prepared for review")
    except ValueError as error:
        print(error)
    else:
        raise AssertionError("Only the owner can report completion")
board.close()
```

</details>